# [Chapter 12 — Two-Group Models, §12.4] The Next-Generation Matrix and $\mathcal{R}_0$ for Two Groups

**Book:** *Essential Considerations for Modeling Epidemics* (Hyman/Qu/Xue, 2026)
**Chapter:** Chapter 12 — Two-Group Models
**Considerations developed:** 12 (basic vs effective $R$ misapplication), 7 (mixing balance)
**Estimated runtime:** ≤ 15 s

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/machyman/hyman2026essential/blob/main/python/notebooks/chapter_12_two_group/02_NGM_two_group_R0.ipynb)


## What this notebook does

Constructs the next-generation matrix $K$ for the two-group $SIR_I$ model and computes $\mathcal{R}_0$ as its dominant eigenvalue. The notebook then compares this against three flawed shortcuts: (a) using the population-mean contact rate $\bar c$, (b) reporting the *arithmetic* mean of group reproductive numbers, and (c) reporting only the high-activity group's $\mathcal{R}_0$. Each shortcut violates Consideration~12 (basic vs effective $R$ misapplication) in a different way.

## Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from shared import book_style, BOOK_COLORS
from shared.seeds import set_seed_chapter_12
from shared.verification import assert_within_tolerance
set_seed_chapter_12()
book_style()


## NGM construction

For the two-group $SIR_I$ at the DFE, the next-generation matrix is
$$K_{ij} = \beta \, c_i \, c_{ij} \, \tau_R \frac{S_i^0}{N_i},$$
and $\mathcal{R}_0 = \rho(K)$, the spectral radius.

In [ ]:
from shared.parameters import baseline_chapter_12
p = baseline_chapter_12()
c1, c2 = p['c1'], p['c2']
N1, N2 = p['N1'], p['N2']
beta, tau_R = p['beta'], p['tau_R']

denom = c1*N1 + c2*N2
M = np.array([[c1*N1, c2*N2], [c1*N1, c2*N2]]) / denom

K = np.array([
    [beta*c1*M[0,0]*tau_R, beta*c1*M[0,1]*tau_R],
    [beta*c2*M[1,0]*tau_R, beta*c2*M[1,1]*tau_R],
])
print('Next-generation matrix K:')
print(K)
eigs = np.linalg.eigvals(K)
R0_NGM = float(np.max(np.real(eigs)))
print(f'\nEigenvalues: {eigs}')
print(f'R_0 (NGM, exact) = {R0_NGM:.4f}')


## Three flawed shortcuts (Consideration~12)

In [ ]:
# Shortcut A: population-mean contact rate
c_bar = c1*N1 + c2*N2
R0_mean = c_bar * beta * tau_R

# Shortcut B: arithmetic mean of group R_0s
R0_g1 = c1 * beta * tau_R
R0_g2 = c2 * beta * tau_R
R0_arith = 0.5*(R0_g1 + R0_g2)

# Shortcut C: high-activity group only
R0_high = R0_g2

print(f"{'method':<35}  {'R_0':>6}  {'rel. error':>12}")
print('-'*60)
for label, val in [
    ('NGM (correct)',          R0_NGM),
    ('mean contact rate',      R0_mean),
    ('arithmetic mean of R_g', R0_arith),
    ('high-activity only',     R0_high),
]:
    err = (val - R0_NGM)/R0_NGM * 100
    print(f'{label:<35}  {val:>6.3f}  {err:>+11.2f}%')


### Why the NGM differs from the mean

Heterogeneity inflates $\mathcal{R}_0$ because the high-activity group both *contracts* and *transmits* more — the $c^2$ weighting in the NGM captures this self-reinforcement. The mean-contact shortcut misses it; the high-only shortcut overshoots because it ignores the lower-activity group's dilution.

In [ ]:
# Verify: NGM must equal exactly (c1^2 N1 + c2^2 N2)/(c1 N1 + c2 N2) * beta * tau_R for proportional mixing
R0_closed = (c1**2*N1 + c2**2*N2)/(c1*N1 + c2*N2) * beta * tau_R
assert_within_tolerance(R0_NGM, R0_closed, abs_tol=1e-10,
    label='NGM eigenvalue matches closed-form proportional-mixing R_0')
print(f'Closed-form proportional-mixing R_0 = {R0_closed:.4f}')
print(f'Matches NGM dominant eigenvalue.')


## What's next

- Chapter 13 applies NGM to bipartite (heterosexual) STI models, where the off-diagonal blocks dominate.
- Chapter 14 applies NGM to vector-host models, recovering the Ross-Macdonald formula.
- Once $\mathcal{R}_0$ is known, Chapter 10's sensitivity machinery tells which contact-mixing parameter to perturb for the largest reduction — the natural lead-in to intervention design.